In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
import requests
from dotenv import load_dotenv

load_dotenv()
GITHUB_TOKEN = os.getenv("GITHUB_API_KEY")
user = os.getenv("USER")

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"TOOL_RESULT:The weather in {city} is sunny and 28°C."

@tool
def get_open_pullrequests(repo: str) -> str:
    """Get pull requests on a repository in github from its name"""
    url = f"https://api.github.com/repos/{user
    }/{repo}/pulls"
    print(repo)
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "X-GitHub-Api-Version": "2026-03-10",
    }
    response = requests.get(url, headers=headers)

    response.raise_for_status()

    prs = response.json()

    return [
        {
            "number": pr["number"],
            "title": pr["title"],
            "body": pr["body"],
            "url": pr["html_url"],
            "author": pr["user"]["login"],
            "state": pr["state"]
        }
        for pr in prs
    ]

model = ChatOllama(
    model="qwen3:4b",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[get_weather,get_open_pullrequests],
    system_prompt="""
        You are a fast assistant.
        Use tools when needed.
        Do not overthink."""
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Get PR raised on CollabWithBatchmates repo?"
        }
    ]
})

print(response["messages"][-1].content)
print(response)


CollabWithBatchmates
There are 2 open pull requests in the `CollabWithBatchmates` repo:

1. **PR #2**: *Feature1* (by `pratham5816`)  
   🔗 [View PR](https://github.com/pratham5816/CollabWithBatchmates/pull/2)

2. **PR #1**: *Amogh changes* (by `pratham5816`)  
   🔗 [View PR](https://github.com/pratham5816/CollabWithBatchmates/pull/1)
{'messages': [HumanMessage(content='Get PR raised on CollabWithBatchmates repo?', additional_kwargs={}, response_metadata={}, id='f8687f52-08d5-46c2-88b7-8750b6cc1179'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-03T21:09:57.824134006Z', 'done': True, 'done_reason': 'stop', 'total_duration': 56202210213, 'load_duration': 1185675, 'prompt_eval_count': 207, 'prompt_eval_duration': 4664630000, 'eval_count': 371, 'eval_duration': 51509352000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a10399-5035-7203-9ec4-795ed54d3a54-0', tool_calls=[{'name': 'get_open